In [ ]:

import pandas as pd

csv_path = "benchmarks/entropy-wave-convergence-2d/entropy-wave-convergence-2d.csv"
tex_path = "benchmarks/entropy-wave-convergence-2d/entropy-wave-convergence-2d.tex"


In [ ]:
# Keep the same method order as the benchmark.
configs = [
    "MUSCL-Hancock",
    "MUSCL-RK3",
    "ZS3",
    "ZS3lazy",
    "MM3/1rev/rtol_1e-1",
    "MM3/1rev/rtol_0",
    "ZS7",
    "ZS7lazy",
    "MM7/1rev/rtol_1e-1",
    "MM7/1rev/rtol_0",
]

labels = {
    "MUSCL-Hancock": "MUSCL-Hancock",
    "MUSCL-RK3": r"MUSCL$+$SSPRK3",
    "ZS3": r"4th-order \textit{a priori}",
    "ZS3lazy": r"4th-order \textit{a priori}, $\overline{\boldsymbol{W}}^{(1)}$",
    "MM3/1rev/rtol_1e-1": r"4th-order \textit{a posteriori}, $\epsilon=10^{-1}$",
    "MM3/1rev/rtol_0": r"4th-order \textit{a posteriori}, $\epsilon=0$",
    "ZS7": r"8th-order \textit{a priori}",
    "ZS7lazy": r"8th-order \textit{a priori}, $\overline{\boldsymbol{W}}^{(1)}$",
    "MM7/1rev/rtol_1e-1": r"8th-order \textit{a posteriori}, $\epsilon=10^{-1}$",
    "MM7/1rev/rtol_0": r"8th-order \textit{a posteriori}, $\epsilon=0$",
}

df = pd.read_csv(csv_path)
df["name"] = pd.Categorical(df["name"], categories=configs, ordered=True)
df = df.sort_values(["name", "N"])


In [ ]:
# Inspect the recorded values before formatting for the manuscript.
table = df.set_index(["name", "N"])[["L1", "L1_rate", "Linf", "Linf_rate"]]
table


In [ ]:
def format_error(value):
    if pd.isna(value):
        return "--"
    mantissa, exponent = f"{value:.2e}".split("e")
    return rf"${mantissa} \times 10^{{{int(exponent)}}}$"


def format_rate(value):
    return "--" if pd.isna(value) else f"{value:.2f}"


# Rates are log(E_coarse / E_fine) / log(N_fine / N_coarse).
# The first resolution and zero-error comparisons have undefined rates.
df_to_table = df.copy()
df_to_table["Method"] = [labels[name] for name in df_to_table["name"]]
for norm in ("L1", "Linf"):
    df_to_table[norm] = df_to_table[norm].map(format_error)
    df_to_table[f"{norm}_rate"] = df_to_table[f"{norm}_rate"].map(format_rate)

manuscript_table = df_to_table.set_index(["Method", "N"])[
    ["L1", "L1_rate", "Linf", "Linf_rate"]
]
manuscript_table.columns = [r"$L^1$ error", "Rate", r"$L^\infty$ error", "Rate"]

# Requires \usepackage{booktabs, amsmath, amssymb} in the manuscript preamble.
latex = manuscript_table.to_latex(
    escape=False,
    column_format="lrrrrr",
    multirow=False,
    caption=(
        r"Two-dimensional entropy-wave convergence at $t=1$. Errors compare "
        r"$v_z$ with its initial state; $L^1$ is the mean absolute error. "
        r"Rates compare successive resolutions within each method."
    ),
    label="tab:entropy-wave-convergence-2d",
)
tex_path.write_text(latex)
print(latex)
